# Basic MNIST Network
Create a very basic neural net, train it, and save to disk.

In [1]:
import torch
import torchvision

In [2]:
from torch import nn

class BasicModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        x = self.layers(x)
        return x

In [3]:
from torchvision.transforms import ToTensor, Compose

class FlattenTensor:
    def __call__(self, x):
        return torch.flatten(x)

transform_fn = Compose([
    ToTensor(),
    FlattenTensor()
])
training_set = torchvision.datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform_fn
)
test_set = torchvision.datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform_fn
)

In [4]:
from torch.utils.data import DataLoader
BATCH_SIZE = 24
training_loader = DataLoader(training_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
# Initialize Model
model = BasicModel()
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(model)

In [ ]:
import trainer
EPOCHS = 100
MIN_LOSS = 0.008
for i in range(EPOCHS):
    print(f"\n---- EPOCH {i + 1} ----")
    loss = trainer.train_one_epoch(model, training_loader, loss_fn, optimizer, print_freq=None)
    print(f"Epoch Loss: {loss:>8f}")
    if loss <= MIN_LOSS:
        print(f"Reached loss of {loss}")
        break

trainer.evaluate_model(model, test_loader, loss_fn, "mps")

In [7]:
# Save the Model
script_model = torch.jit.script(model)
torch.jit.save(script_model, "torchscript-models/basic_model.pt")
